# Gold: lab_test_trends

Builds `<catalog>.gold.lab_test_trends` (one row per test_name per calendar month) from silver lab_results and calendar, per the STTM `sttm_silver_to_gold.xlsx`. Includes abnormal/critical rates and a month-over-month percentage-point change in abnormal rate via LAG. The STTM's `calendar.date_key` maps to the actual INT column `calendar.datekey`.

In [0]:
# Catalog placeholder from the STTM (<catalog>); override via widget or job parameter
dbutils.widgets.text("catalog", "adb_caresync", "Catalog")
print("Using catalog:", dbutils.widgets.get("catalog"))

In [0]:
%sql
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.gold.lab_test_trends')
COMMENT 'Gold: one row per lab test type per calendar month with abnormal/critical rates and month-over-month trend (STTM silver to gold v1.0)'
AS
WITH monthly AS (
  SELECT
    test_name,
    year(test_date)                          AS year,
    month(test_date)                         AS month,
    COUNT(*)                                 AS total_tests,
    COUNT_IF(result_status = 'Abnormal')     AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')     AS critical_test_count
  FROM IDENTIFIER(:catalog || '.silver.lab_results')
  GROUP BY test_name, year(test_date), month(test_date)
),
rates AS (
  SELECT
    m.*,
    CAST(round(m.abnormal_test_count * 100.0 / m.total_tests, 2) AS DECIMAL(5,2)) AS abnormal_rate_pct,
    CAST(round(m.critical_test_count * 100.0 / m.total_tests, 2) AS DECIMAL(5,2)) AS critical_rate_pct
  FROM monthly m
),
with_lag AS (
  SELECT
    r.*,
    LAG(r.abnormal_rate_pct) OVER (PARTITION BY r.test_name ORDER BY r.year, r.month) AS prev_month_abnormal_rate_pct
  FROM rates r
)
SELECT
  w.test_name                                                            AS test_name,
  CAST(w.year AS INT)                                                    AS year,
  CAST(w.month AS INT)                                                   AS month,
  c.month_name                                                           AS month_name,
  c.quarter_name                                                         AS quarter_name,
  CAST(w.total_tests AS INT)                                             AS total_tests,
  CAST(w.abnormal_test_count AS INT)                                     AS abnormal_test_count,
  CAST(w.critical_test_count AS INT)                                     AS critical_test_count,
  w.abnormal_rate_pct                                                    AS abnormal_rate_pct,
  w.critical_rate_pct                                                    AS critical_rate_pct,
  CAST(w.prev_month_abnormal_rate_pct AS DECIMAL(5,2))                   AS prev_month_abnormal_rate_pct,
  CAST(w.abnormal_rate_pct - w.prev_month_abnormal_rate_pct AS DECIMAL(6,2)) AS abnormal_rate_change_pct
FROM with_lag w
LEFT JOIN IDENTIFIER(:catalog || '.silver.calendar') c
  ON c.datekey = CAST(date_format(make_date(w.year, w.month, 1), 'yyyyMMdd') AS INT)

In [0]:
import json

# Count the number of rows processed in this batch (rows written to the gold table)
target_table = f"{dbutils.widgets.get('catalog')}.gold.lab_test_trends"
updated_df = spark.table(target_table)
record_count = updated_df.count()
print(f"{target_table}: {record_count} rows")

# End the notebook and return the record count to the calling job or notebook
dbutils.notebook.exit(json.dumps({"record_count": record_count}))